# LIFE on Google Colab — PolitiFact++ (binary MF-vs-MR, LLaMA2-7B)

Faithful reproduction of the paper's setup: binary fake/real over the **LLM pair** (MF=fake, MR=real), with **LLaMA2-7B** as the reconstruction model. End-to-end: **convert → key-sentence extraction → concatenate → features → train**.

**Before you start:**
1. Set the Colab runtime to **GPU** — an **A100** is needed for LLaMA2-7B (Runtime → Change runtime type).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.
4. Step 3 uses the **ungated** `NousResearch/Llama-2-7b-hf` mirror by default — no HF token needed. (The official `meta-llama/Llama-2-7b-hf` is gated and requires an approved access request + token.)

Scope: **PolitiFact++** only (~229 LLM-pair articles: 97 fake + 132 real). VLPFN is excluded (its text has no punctuation, so sentence splitting cannot work). GossipCop++ is far heavier; try it only after this works.

In [1]:
# Confirm a GPU is attached
!nvidia-smi

Wed Jun 10 19:39:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

POLITIFACT_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/PolitiFact++'

# Paper's binary task: LLM pair only (MF=fake, MR=real), reconstructed with LLaMA2-7B.
OUTPUT_BIN     = f'{PROJECT_DIR}/dataset/output_bin'        # MF_fake.jsonl + MR_true.jsonl
KEY_SENT       = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_top10.jsonl'
BERT_CKPT      = f'{PROJECT_DIR}/dataset/bert_bin.pt'       # fresh extractor for MF-vs-MR
FEATURES_LLAMA = f'{PROJECT_DIR}/dataset/features_llama'
TRAIN_PATH     = f'{PROJECT_DIR}/dataset/train_bin.jsonl'
TEST_PATH      = f'{PROJECT_DIR}/dataset/test_bin.jsonl'

print('cwd:', os.getcwd())
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))

cwd: /content/drive/MyDrive/LIFE
PolitiFact++ found: True


In [4]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.7/644.7 kB 11.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.3/217.3 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymc 5.28.5 requires rich>=13.7.1, but you have rich 11.2.0 which is incompatible.
bigframes 2.41.0 requires rich<14,>=12.4.4, but you have rich 11.2.0 which is incompatible.


In [5]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## HuggingFace login (optional)
Step 3 defaults to the **ungated** `NousResearch/Llama-2-7b-hf` mirror, so **no token is needed — you can skip this cell**. Only run it if you switch Step 3 to the official gated `meta-llama/Llama-2-7b-hf` (which also requires an approved access request).

In [6]:
# LLaMA-2 is gated on HuggingFace. First accept the license at
# https://huggingface.co/meta-llama/Llama-2-7b-hf, then run this cell and paste an
# access token from https://huggingface.co/settings/tokens
# (or replace with: login(token="hf_xxx")).
# from huggingface_hub import login
# login()

## Step 0 — Convert PolitiFact++ to the binary LLM-pair JSONL
`--subset llm` emits only `MF_fake.jsonl` (97, fake) and `MR_true.jsonl` (132, real) — the paper's binary task. HF/HR (human-written) are not used.

In [7]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_BIN}" --subset llm

MF.json -> MF_fake.jsonl: 97 records (label=gpt3.5_fake)
MR.json -> MR_true.jsonl: 132 records (label=gpt3.5_true)


## Step 1 — Key-sentence extraction (top-10)
Trains a **fresh** BERT fake/real classifier on MF-vs-MR (saved to `BERT_CKPT`), then keeps the **top-10** most impactful sentences per article (paper's k for PolitiFact++). This is the slowest step (a forward pass per sentence per article).

In [8]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_BIN}" --output_file "{KEY_SENT}" --top_k 10 --model_path "{BERT_CKPT}" --gpu 0

tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 202kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 10.2MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 31.8MB/s]
config.json: 100% 570/570 [00:00<00:00, 3.88MB/s]
model.safetensors: 100% 440M/440M [00:01<00:00, 253MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 6322.23it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 

## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_BIN` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_BIN` in place** — re-run Step 0 first if you need to reset them.

In [9]:
!python dataset/2_concate.py --folder_path "{OUTPUT_BIN}" --important_sentences_file "{KEY_SENT}"

所有 .jsonl 文件已成功更新。


## Step 3 — Reconstruction probabilities with LLaMA2-7B
The paper's reconstruction model. A malicious prompt is prepended and LLaMA2-7B's per-token log-likelihoods over the key fragments form the "linguistic fingerprint" features. Loads in bfloat16 (~14 GB; needs the A100) and downloads ~13 GB on first run. Writes one feature JSONL per input file into `FEATURES_LLAMA`.

In [10]:
# meta-llama/Llama-2-7b-hf is gated (needs Meta approval). NousResearch/Llama-2-7b-hf is an
# ungated mirror of the SAME weights/tokenizer — no token needed. Swap back to the official
# repo if/when your access request is approved.
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_LLAMA}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

device: cuda | model: NousResearch/Llama-2-7b-hf | dtype: bfloat16 | scorer: llama
config.json: 100% 583/583 [00:00<00:00, 3.15MB/s]
tokenizer_config.json: 100% 746/746 [00:00<00:00, 4.74MB/s]
tokenizer.model: 100% 500k/500k [00:00<00:00, 1.23MB/s]
tokenizer.json: 100% 1.84M/1.84M [00:00<00:00, 45.0MB/s]
special_tokens_map.json: 100% 435/435 [00:00<00:00, 3.38MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 26.8k/26.8k [00:00<00:00, 77.9MB/s]
Fetching 2 files: 100% 2/2 [00:36<00:00, 18.15s/it]
Download complete: 100% 13.5G/13.5G [00:36<00:00, 371MB/s]
Loading weights: 100% 291/291 [00:02<00:00, 110.68it/s]
generation_config.json: 100% 200/200 [00:00<00:00, 1.20MB/s]
input file:/content/drive/MyDrive/LIFE/dataset/output_bin/MF_fake.jsonl, length:97
  0% 0/97 [00:00<?, ?it/s]0 57
58 122
124 202
206 280
282 306
310 350
352 436
438 484
486 538
542 578
580 648
  1% 1/97 [00:00<01:05,  1.47it/s]0 57
58 122
124 158
161 197
199 247
250 2

## Step 4 — Train the classifier (binary)
Splits `FEATURES_LLAMA` into train/test and trains the Transformer+CRF classifier for **50 epochs** on the binary MF-vs-MR task. Paper target for PolitiFact++: **Acc 0.900 / F1 0.882**.

In [11]:
!python LIFE_train/train.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --model Transformer \
  --num_train_epochs 50

Log INFO: split dataset...
********************************
The overall data sources:
['MF_fake.jsonl', 'MR_true.jsonl']
100% 183/183 [00:00<00:00, 3766.80it/s]
100% 46/46 [00:00<00:00, 3415.56it/s]

The number of train dataset: 183
The number of test  dataset: 46
********************************
100% 183/183 [00:00<00:00, 777.97it/s]
100% 46/46 [00:00<00:00, 7858.02it/s]
--------------------------------classify--------------------------------
Log INFO: do train...
Epoch:   0% 0/50 [00:00<?, ?it/s]
Iteration:   0% 0/6 [00:00<?, ?it/s]
Iteration:  17% 1/6 [00:01<00:05,  1.20s/it]
Iteration:  33% 2/6 [00:01<00:02,  1.76it/s]
Iteration:  50% 3/6 [00:01<00:01,  2.72it/s]
Iteration:  67% 4/6 [00:01<00:00,  3.74it/s]
Iteration: 100% 6/6 [00:01<00:00,  3.38it/s]
epoch 1: train_loss 2.7130055824915567

Iteration:   0% 0/2 [00:00<?, ?it/s]
Iteration:  50% 1/2 [00:00<00:00,  1.59it/s]
Iteration: 100% 2/2 [00:00<00:00,  2.05it/s]
******** Evalation ********
Accuracy: 51.9
Macro F1 Score: 45.2
Pre

## Notes / troubleshooting
- **HF gating**: Step 3 defaults to the ungated `NousResearch/Llama-2-7b-hf` mirror (no token). If you switch to the official `meta-llama` repo and hit a 403 "gated repo", your access request hasn't been approved yet.
- **fastNLP**: if Step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `BERT_CKPT` (step 1) and `linear_en.pt` (step 4) are written under `PROJECT_DIR` on Drive, so they survive disconnects.
- **NaN features**: if Step 3 prints NaN/inf, switch Step 3 to `--dtype float32` (fits the 40 GB A100).
- **Re-runs**: Step 2 mutates `OUTPUT_BIN` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.